# Landsat-9 TIR Super-Resolution + Colorization  
## Correct Full Model Notebook — Original Sensor Values Only

This notebook implements the complete pipeline we discussed:

1. Raw dataset sanity check  
2. Train-set preprocessing statistics from original sensor arrays  
3. CNN thermal super-resolution  
4. CNN U-Net thermal-to-RGB colorization  
5. Pix2Pix GAN colorization  
6. Tiny Transformer colorization  
7. Evaluation metrics  
8. Prediction visualization  
9. Final two-stage inference for the hackathon/common dataset  

## Critical organizer constraint

**Use original sensor values, not visualization-processed values.**

This notebook follows that rule:

- Model input comes from `.npy` arrays directly.
- Thermal values are numerically normalized for training.
- RGB/reference values are numerically scaled using train-set statistics.
- `matplotlib`, colormaps, percentile stretch, PNG previews, and display functions are used only for visualization.
- No pseudo-colored image is ever used as model input.

## Confirmed dataset format

```text
SR:
TIR 200m  (256, 256)      -> TIR 100m  (512, 512)

Colorization:
TIR 100m  (256, 256)      -> RGB 100m  (3, 256, 256)
```

# Fast Run Order

If time is short, run only these first:

```text
1. Setup
2. Dataset sanity check
3. Compute/load preprocessing stats
4. Build dataloaders
5. CNN SR model shape test
6. Train CNN SR
7. Visualize CNN SR
8. CNN Color U-Net shape test
9. Train CNN Color U-Net
10. Visualize colorization
11. Final two-stage inference
```

Run GAN and Transformer after the CNN pipeline is working.

For a quick run:

```python
SR_EPOCHS = 5
COLOR_CNN_EPOCHS = 5
GAN_EPOCHS = 2
VIT_EPOCHS = 2
```

For better final results:

```python
SR_EPOCHS = 20
COLOR_CNN_EPOCHS = 25
GAN_EPOCHS = 10
VIT_EPOCHS = 15
```

In [ ]:
# ============================================================
# 1. Setup
# ============================================================

import os, glob, json, math, time, random
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive mount skipped/not available:", e)

# Confirmed dataset path.
DATASET_ROOT = "/content/drive/MyDrive/landsat_india_200/dataset_current_repo_format"

# If running on Kaggle/local, edit this:
# DATASET_ROOT = "/kaggle/input/your-dataset/dataset_current_repo_format"

assert os.path.exists(DATASET_ROOT), f"Dataset root not found: {DATASET_ROOT}"
print("Using dataset root:", DATASET_ROOT)

SAVE_DIR = "/content/drive/MyDrive/landsat_india_200/checkpoints_correct_full"
try:
    os.makedirs(SAVE_DIR, exist_ok=True)
except Exception:
    SAVE_DIR = "./checkpoints_correct_full"
    os.makedirs(SAVE_DIR, exist_ok=True)
print("Saving to:", SAVE_DIR)

BATCH_SIZE_SR = 4
BATCH_SIZE_COLOR = 4
NUM_WORKERS = 2

# Change these for quick/final runs.
SR_EPOCHS = 10
COLOR_CNN_EPOCHS = 15
GAN_EPOCHS = 10
VIT_EPOCHS = 15

# For quick debug:
# SR_EPOCHS = 2
# COLOR_CNN_EPOCHS = 2
# GAN_EPOCHS = 1
# VIT_EPOCHS = 1

In [ ]:
# ============================================================
# 2. Dataset sanity check
# ============================================================

def list_npy(folder: str) -> List[str]:
    return sorted(glob.glob(os.path.join(folder, "*.npy")))

def count_npy(folder: str) -> int:
    return len(list_npy(folder))

folder_map = {
    "sr_train_x": "sr/train/tir_200m",
    "sr_train_y": "sr/train/tir_100m",
    "sr_val_x": "sr/val/tir_200m",
    "sr_val_y": "sr/val/tir_100m",
    "sr_test_x": "sr/test/tir_200m",
    "sr_test_y": "sr/test/tir_100m",
    "color_train_x": "colorization/train/tir_100m",
    "color_train_y": "colorization/train/rgb_100m",
    "color_val_x": "colorization/val/tir_100m",
    "color_val_y": "colorization/val/rgb_100m",
    "color_test_x": "colorization/test/tir_100m",
    "color_test_y": "colorization/test/rgb_100m",
}

print("\nFILE COUNTS")
for name, rel in folder_map.items():
    folder = os.path.join(DATASET_ROOT, rel)
    print(f"{name:15s} {count_npy(folder):5d}  {folder}")

def summarize_array(arr: np.ndarray) -> Dict[str, object]:
    return {
        "shape": tuple(arr.shape),
        "dtype": str(arr.dtype),
        "min": float(np.nanmin(arr)),
        "max": float(np.nanmax(arr)),
        "mean": float(np.nanmean(arr)),
        "std": float(np.nanstd(arr)),
        "finite": bool(np.isfinite(arr).all()),
    }

def check_pair(task: str, split: str, x_folder: str, y_folder: str, expected_x_shape=None, expected_y_shape=None):
    x_dir = os.path.join(DATASET_ROOT, task, split, x_folder)
    y_dir = os.path.join(DATASET_ROOT, task, split, y_folder)

    x_files = list_npy(x_dir)
    y_files = list_npy(y_dir)

    x_names = set(os.path.basename(p) for p in x_files)
    y_names = set(os.path.basename(p) for p in y_files)

    print(f"\nChecking {task}/{split}")
    print("x files:", len(x_files))
    print("y files:", len(y_files))
    print("missing y:", len(x_names - y_names))
    print("missing x:", len(y_names - x_names))

    assert len(x_files) > 0, f"No files found in {x_dir}"
    assert x_names == y_names, f"Pair mismatch in {task}/{split}"

    sample_name = random.choice(sorted(list(x_names)))
    x = np.load(os.path.join(x_dir, sample_name)).astype(np.float32)
    y = np.load(os.path.join(y_dir, sample_name)).astype(np.float32)

    print("sample:", sample_name)
    print("x:", summarize_array(x))
    print("y:", summarize_array(y))

    if expected_x_shape is not None:
        assert tuple(x.shape) == tuple(expected_x_shape), f"Expected x shape {expected_x_shape}, got {x.shape}"
    if expected_y_shape is not None:
        assert tuple(y.shape) == tuple(expected_y_shape), f"Expected y shape {expected_y_shape}, got {y.shape}"

    assert np.isfinite(x).all(), f"Non-finite x values in {sample_name}"
    assert np.isfinite(y).all(), f"Non-finite y values in {sample_name}"

for split in ["train", "val", "test"]:
    check_pair("sr", split, "tir_200m", "tir_100m", (256, 256), (512, 512))
    check_pair("colorization", split, "tir_100m", "rgb_100m", (256, 256), (3, 256, 256))

print("\nDataset sanity check passed.")

# 3. Preprocessing Statistics From Original Train Values

This computes training statistics from original `.npy` arrays only.

It saves:

```text
preprocess_stats.json
```

This file must be used during common-dataset/finale inference.

In [ ]:
# ============================================================
# 3. Compute/load preprocessing stats
# ============================================================

STATS_PATH = os.path.join(SAVE_DIR, "preprocess_stats.json")

def safe_load_npy(path: str) -> np.ndarray:
    arr = np.load(path).astype(np.float32)
    if not np.isfinite(arr).all():
        arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    return arr

def compute_tir_mean_std_from_train(root: str) -> Tuple[float, float]:
    paths = []
    paths += list_npy(os.path.join(root, "sr", "train", "tir_200m"))
    paths += list_npy(os.path.join(root, "sr", "train", "tir_100m"))
    paths += list_npy(os.path.join(root, "colorization", "train", "tir_100m"))
    assert len(paths) > 0, "No TIR training files found."

    total_sum = 0.0
    total_sq = 0.0
    total_count = 0

    for p in tqdm(paths, desc="Computing TIR stats"):
        arr = safe_load_npy(p)
        arr = arr[np.isfinite(arr)]
        total_sum += float(arr.sum())
        total_sq += float((arr ** 2).sum())
        total_count += int(arr.size)

    mean = total_sum / total_count
    var = total_sq / total_count - mean ** 2
    std = math.sqrt(max(var, 1e-8))
    return float(mean), float(std)

def compute_rgb_min_max_from_train(root: str) -> Tuple[List[float], List[float]]:
    paths = list_npy(os.path.join(root, "colorization", "train", "rgb_100m"))
    assert len(paths) > 0, "No RGB training files found."

    band_min = np.array([np.inf, np.inf, np.inf], dtype=np.float64)
    band_max = np.array([-np.inf, -np.inf, -np.inf], dtype=np.float64)

    for p in tqdm(paths, desc="Computing RGB stats"):
        arr = safe_load_npy(p)
        if arr.ndim == 3 and arr.shape[0] == 3:
            chw = arr
        elif arr.ndim == 3 and arr.shape[-1] == 3:
            chw = np.moveaxis(arr, -1, 0)
        else:
            raise ValueError(f"Unexpected RGB shape {arr.shape} in {p}")

        for c in range(3):
            band = chw[c]
            band = band[np.isfinite(band)]
            band_min[c] = min(band_min[c], float(band.min()))
            band_max[c] = max(band_max[c], float(band.max()))

    return band_min.tolist(), band_max.tolist()

if os.path.exists(STATS_PATH):
    with open(STATS_PATH, "r") as f:
        PREPROCESS_STATS = json.load(f)
    print("Loaded preprocessing stats:", STATS_PATH)
else:
    tir_mean, tir_std = compute_tir_mean_std_from_train(DATASET_ROOT)
    rgb_min, rgb_max = compute_rgb_min_max_from_train(DATASET_ROOT)
    PREPROCESS_STATS = {
        "tir_mean": tir_mean,
        "tir_std": tir_std,
        "rgb_min": rgb_min,
        "rgb_max": rgb_max,
        "source": "original_train_sensor_values",
        "note": "No visualization stretching or colormap values used."
    }
    with open(STATS_PATH, "w") as f:
        json.dump(PREPROCESS_STATS, f, indent=2)
    print("Saved preprocessing stats:", STATS_PATH)

TIR_MEAN = float(PREPROCESS_STATS["tir_mean"])
TIR_STD = float(PREPROCESS_STATS["tir_std"])
RGB_MIN = np.array(PREPROCESS_STATS["rgb_min"], dtype=np.float32).reshape(3, 1, 1)
RGB_MAX = np.array(PREPROCESS_STATS["rgb_max"], dtype=np.float32).reshape(3, 1, 1)

print(json.dumps(PREPROCESS_STATS, indent=2))

In [ ]:
# ============================================================
# 4. Preprocessing functions
# ============================================================

def normalize_tir_tensor(x: torch.Tensor) -> torch.Tensor:
    return (x - TIR_MEAN) / (TIR_STD + 1e-8)

def denormalize_tir_tensor(x: torch.Tensor) -> torch.Tensor:
    return x * (TIR_STD + 1e-8) + TIR_MEAN

def rgb_min_tensor(device=None, dtype=torch.float32):
    return torch.tensor(RGB_MIN, dtype=dtype, device=device)

def rgb_max_tensor(device=None, dtype=torch.float32):
    return torch.tensor(RGB_MAX, dtype=dtype, device=device)

def normalize_rgb_tensor(y: torch.Tensor) -> torch.Tensor:
    mn = rgb_min_tensor(device=y.device, dtype=y.dtype)
    mx = rgb_max_tensor(device=y.device, dtype=y.dtype)
    if y.ndim == 4:
        mn = mn.unsqueeze(0)
        mx = mx.unsqueeze(0)
    return torch.clamp((y - mn) / (mx - mn + 1e-8), 0.0, 1.0)

def denormalize_rgb_tensor(y: torch.Tensor) -> torch.Tensor:
    mn = rgb_min_tensor(device=y.device, dtype=y.dtype)
    mx = rgb_max_tensor(device=y.device, dtype=y.dtype)
    if y.ndim == 4:
        mn = mn.unsqueeze(0)
        mx = mx.unsqueeze(0)
    return y * (mx - mn + 1e-8) + mn

def ensure_rgb_chw(arr: np.ndarray) -> np.ndarray:
    arr = arr.astype(np.float32)
    if arr.ndim == 3 and arr.shape[0] == 3:
        return arr
    if arr.ndim == 3 and arr.shape[-1] == 3:
        return np.moveaxis(arr, -1, 0)
    raise ValueError(f"Expected RGB as 3,H,W or H,W,3 but got {arr.shape}")

print("Preprocessing functions ready.")

In [ ]:
# ============================================================
# 5. Dataset classes and dataloaders
# ============================================================

class LandsatSRDataset(Dataset):
    def __init__(self, root: str, split: str, augment: bool = False):
        self.root = root
        self.split = split
        self.augment = augment
        self.x_dir = os.path.join(root, "sr", split, "tir_200m")
        self.y_dir = os.path.join(root, "sr", split, "tir_100m")
        self.names = sorted(os.path.basename(p) for p in list_npy(self.x_dir))
        assert len(self.names) > 0, f"No SR files found in {self.x_dir}"
        for name in self.names:
            assert os.path.exists(os.path.join(self.y_dir, name)), f"Missing SR target: {name}"

    def __len__(self):
        return len(self.names)

    def _augment(self, x, y):
        if random.random() < 0.5:
            x = torch.flip(x, dims=[2])
            y = torch.flip(y, dims=[2])
        if random.random() < 0.5:
            x = torch.flip(x, dims=[1])
            y = torch.flip(y, dims=[1])
        k = random.randint(0, 3)
        if k > 0:
            x = torch.rot90(x, k, dims=[1, 2])
            y = torch.rot90(y, k, dims=[1, 2])
        return x, y

    def __getitem__(self, idx):
        name = self.names[idx]
        x_np = safe_load_npy(os.path.join(self.x_dir, name))
        y_np = safe_load_npy(os.path.join(self.y_dir, name))

        assert x_np.shape == (256, 256), f"Bad SR x shape {x_np.shape} for {name}"
        assert y_np.shape == (512, 512), f"Bad SR y shape {y_np.shape} for {name}"

        x = torch.from_numpy(x_np).unsqueeze(0)
        y = torch.from_numpy(y_np).unsqueeze(0)

        x = normalize_tir_tensor(x)
        y = normalize_tir_tensor(y)

        if self.augment:
            x, y = self._augment(x, y)

        return x, y, name


class LandsatColorDataset(Dataset):
    def __init__(self, root: str, split: str, augment: bool = False):
        self.root = root
        self.split = split
        self.augment = augment
        self.x_dir = os.path.join(root, "colorization", split, "tir_100m")
        self.y_dir = os.path.join(root, "colorization", split, "rgb_100m")
        self.names = sorted(os.path.basename(p) for p in list_npy(self.x_dir))
        assert len(self.names) > 0, f"No color files found in {self.x_dir}"
        for name in self.names:
            assert os.path.exists(os.path.join(self.y_dir, name)), f"Missing color target: {name}"

    def __len__(self):
        return len(self.names)

    def _augment(self, x, y):
        if random.random() < 0.5:
            x = torch.flip(x, dims=[2])
            y = torch.flip(y, dims=[2])
        if random.random() < 0.5:
            x = torch.flip(x, dims=[1])
            y = torch.flip(y, dims=[1])
        k = random.randint(0, 3)
        if k > 0:
            x = torch.rot90(x, k, dims=[1, 2])
            y = torch.rot90(y, k, dims=[1, 2])
        return x, y

    def __getitem__(self, idx):
        name = self.names[idx]
        x_np = safe_load_npy(os.path.join(self.x_dir, name))
        y_np = ensure_rgb_chw(safe_load_npy(os.path.join(self.y_dir, name)))

        assert x_np.shape == (256, 256), f"Bad color x shape {x_np.shape} for {name}"
        assert y_np.shape == (3, 256, 256), f"Bad color y shape {y_np.shape} for {name}"

        x = torch.from_numpy(x_np).unsqueeze(0)
        y = torch.from_numpy(y_np)

        x = normalize_tir_tensor(x)
        y = normalize_rgb_tensor(y)

        if self.augment:
            x, y = self._augment(x, y)

        return x, y, name


def make_loader(ds, batch_size, shuffle):
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == "cuda"),
        drop_last=False
    )

sr_train_ds = LandsatSRDataset(DATASET_ROOT, "train", augment=True)
sr_val_ds = LandsatSRDataset(DATASET_ROOT, "val", augment=False)
sr_test_ds = LandsatSRDataset(DATASET_ROOT, "test", augment=False)

color_train_ds = LandsatColorDataset(DATASET_ROOT, "train", augment=True)
color_val_ds = LandsatColorDataset(DATASET_ROOT, "val", augment=False)
color_test_ds = LandsatColorDataset(DATASET_ROOT, "test", augment=False)

sr_train_loader = make_loader(sr_train_ds, BATCH_SIZE_SR, True)
sr_val_loader = make_loader(sr_val_ds, BATCH_SIZE_SR, False)
sr_test_loader = make_loader(sr_test_ds, BATCH_SIZE_SR, False)

color_train_loader = make_loader(color_train_ds, BATCH_SIZE_COLOR, True)
color_val_loader = make_loader(color_val_ds, BATCH_SIZE_COLOR, False)
color_test_loader = make_loader(color_test_ds, BATCH_SIZE_COLOR, False)

xb, yb, names = next(iter(sr_train_loader))
print("SR batch:", xb.shape, yb.shape, names[:2])
xb, yb, names = next(iter(color_train_loader))
print("Color batch:", xb.shape, yb.shape, names[:2])

assert next(iter(sr_train_loader))[0].shape[1:] == (1, 256, 256)
assert next(iter(sr_train_loader))[1].shape[1:] == (1, 512, 512)
assert next(iter(color_train_loader))[0].shape[1:] == (1, 256, 256)
assert next(iter(color_train_loader))[1].shape[1:] == (3, 256, 256)
print("Dataloaders ready.")

In [ ]:
# ============================================================
# 6. Visualization helpers
# ============================================================
# IMPORTANT: These are only for display, never for training.

def stretch_for_display(img: np.ndarray, low=2, high=98) -> np.ndarray:
    img = np.asarray(img, dtype=np.float32)
    finite = np.isfinite(img)
    if finite.sum() == 0:
        return np.zeros_like(img)
    p_low, p_high = np.percentile(img[finite], [low, high])
    return np.clip((img - p_low) / (p_high - p_low + 1e-8), 0, 1)

def rgb_chw_to_display(rgb_chw: np.ndarray) -> np.ndarray:
    rgb_hwc = np.moveaxis(rgb_chw.astype(np.float32), 0, -1)
    return stretch_for_display(rgb_hwc)

def show_sr_sample(ds, idx=0):
    x, y, name = ds[idx]
    x_raw = denormalize_tir_tensor(x)[0].numpy()
    y_raw = denormalize_tir_tensor(y)[0].numpy()

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(x_raw, cmap="inferno")
    plt.title(f"Input TIR 200m\n{x_raw.shape}")
    plt.colorbar()
    plt.axis("off")
    plt.subplot(1, 2, 2)
    plt.imshow(y_raw, cmap="inferno")
    plt.title(f"Target TIR 100m\n{y_raw.shape}")
    plt.colorbar()
    plt.axis("off")
    plt.show()

    print("File:", name)
    print("Input raw range:", x_raw.min(), x_raw.max())
    print("Target raw range:", y_raw.min(), y_raw.max())

def show_color_sample(ds, idx=0):
    x, y, name = ds[idx]
    x_raw = denormalize_tir_tensor(x)[0].numpy()
    y_raw = denormalize_rgb_tensor(y).numpy()
    y_disp = rgb_chw_to_display(y_raw)

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(x_raw, cmap="inferno")
    plt.title(f"Input TIR 100m\n{x_raw.shape}")
    plt.colorbar()
    plt.axis("off")
    plt.subplot(1, 2, 2)
    plt.imshow(y_disp)
    plt.title("Target RGB display\n(stretch only)")
    plt.axis("off")
    plt.show()

    print("File:", name)
    print("TIR raw range:", x_raw.min(), x_raw.max())
    print("RGB original-scale range:", y_raw.min(), y_raw.max())

show_sr_sample(sr_train_ds, idx=0)
show_color_sample(color_train_ds, idx=0)

# 7. Model A — CNN Super-Resolution

Objective:

```text
TIR 200m 256×256 -> TIR 100m 512×512
```

The model first bicubic-upsamples to 512×512, then learns a residual correction.

In [ ]:
# ============================================================
# 7. CNN Super-Resolution model
# ============================================================

class ResidualBlock(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        identity = x
        out = self.act(self.conv1(x))
        out = self.conv2(out)
        return identity + out

class SimpleSRNet(nn.Module):
    def __init__(self, channels: int = 64, num_blocks: int = 6):
        super().__init__()
        self.head = nn.Conv2d(1, channels, 3, padding=1)
        self.body = nn.Sequential(*[ResidualBlock(channels) for _ in range(num_blocks)])
        self.tail = nn.Conv2d(channels, 1, 3, padding=1)

    def forward(self, x):
        x_up = F.interpolate(x, scale_factor=2, mode="bicubic", align_corners=False)
        feat = F.relu(self.head(x_up))
        feat = self.body(feat)
        residual = self.tail(feat)
        return x_up + residual

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

sr_model = SimpleSRNet(channels=64, num_blocks=6).to(DEVICE)
xb, yb, names = next(iter(sr_train_loader))
xb, yb = xb.to(DEVICE), yb.to(DEVICE)
with torch.no_grad():
    pred = sr_model(xb)

print("Input shape :", xb.shape)
print("Target shape:", yb.shape)
print("Pred shape  :", pred.shape)
print("Parameters  :", count_params(sr_model))
assert pred.shape == yb.shape

# 8. Model B — CNN U-Net Colorization

Objective:

```text
TIR 100m 256×256 -> RGB 100m 3×256×256
```

In [ ]:
# ============================================================
# 8. CNN U-Net colorization model
# ============================================================

class ConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, use_bn: bool = True):
        super().__init__()
        layers = [nn.Conv2d(in_channels, out_channels, 3, padding=1)]
        if use_bn:
            layers.append(nn.BatchNorm2d(out_channels))
        layers.append(nn.ReLU(inplace=True))
        layers.append(nn.Conv2d(out_channels, out_channels, 3, padding=1))
        if use_bn:
            layers.append(nn.BatchNorm2d(out_channels))
        layers.append(nn.ReLU(inplace=True))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

class ColorUNet(nn.Module):
    def __init__(self, base: int = 32):
        super().__init__()
        self.pool = nn.MaxPool2d(2)

        self.enc1 = ConvBlock(1, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.enc3 = ConvBlock(base * 2, base * 4)
        self.enc4 = ConvBlock(base * 4, base * 8)

        self.bottleneck = ConvBlock(base * 8, base * 16)

        self.up4 = nn.ConvTranspose2d(base * 16, base * 8, 2, stride=2)
        self.dec4 = ConvBlock(base * 16, base * 8)
        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.dec3 = ConvBlock(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = ConvBlock(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = ConvBlock(base * 2, base)

        self.out = nn.Conv2d(base, 3, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b = self.bottleneck(self.pool(e4))

        d4 = self.up4(b)
        d4 = self.dec4(torch.cat([d4, e4], dim=1))
        d3 = self.up3(d4)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return torch.sigmoid(self.out(d1))

color_cnn_model = ColorUNet(base=32).to(DEVICE)
xb, yb, names = next(iter(color_train_loader))
xb, yb = xb.to(DEVICE), yb.to(DEVICE)
with torch.no_grad():
    pred = color_cnn_model(xb)

print("Input shape :", xb.shape)
print("Target shape:", yb.shape)
print("Pred shape  :", pred.shape)
print("Parameters  :", count_params(color_cnn_model))
assert pred.shape == yb.shape

In [ ]:
# ============================================================
# 9. Shared training and evaluation utilities
# ============================================================

def psnr_from_mse(mse: float, data_range: float = 1.0) -> float:
    mse = max(float(mse), 1e-12)
    return 20.0 * math.log10(data_range) - 10.0 * math.log10(mse)

def save_checkpoint(model, path, model_name, task, history, extra=None):
    payload = {
        "model_state_dict": model.state_dict(),
        "model_name": model_name,
        "task": task,
        "history": history,
        "preprocess_stats": PREPROCESS_STATS,
        "created_time": time.strftime("%Y-%m-%d %H:%M:%S")
    }
    if extra:
        payload.update(extra)
    torch.save(payload, path)

def load_model_checkpoint(model, path, strict=True):
    ckpt = torch.load(path, map_location=DEVICE)
    state = ckpt.get("model_state_dict", ckpt)
    model.load_state_dict(state, strict=strict)
    model.to(DEVICE)
    model.eval()
    return model

def evaluate_sr(model, loader):
    model.eval()
    total_abs_norm = 0.0
    total_sq_norm = 0.0
    total_pixels = 0

    with torch.no_grad():
        for x, y, names in tqdm(loader, desc="Evaluating SR", leave=False):
            x, y = x.to(DEVICE), y.to(DEVICE)
            pred = model(x)
            diff = pred - y
            total_abs_norm += diff.abs().sum().item()
            total_sq_norm += (diff ** 2).sum().item()
            total_pixels += y.numel()

    l1_norm = total_abs_norm / total_pixels
    mse_norm = total_sq_norm / total_pixels
    mae_kelvin = l1_norm * TIR_STD
    rmse_kelvin = math.sqrt(mse_norm) * TIR_STD
    psnr_kelvin = psnr_from_mse(rmse_kelvin ** 2, data_range=40.0)

    return {
        "l1_normalized": l1_norm,
        "mse_normalized": mse_norm,
        "mae_kelvin": mae_kelvin,
        "rmse_kelvin": rmse_kelvin,
        "psnr_kelvin_range40": psnr_kelvin
    }

def evaluate_color(model, loader):
    model.eval()
    total_abs_norm = 0.0
    total_sq_norm = 0.0
    total_pixels = 0
    total_abs_orig = 0.0
    total_sq_orig = 0.0

    with torch.no_grad():
        for x, y, names in tqdm(loader, desc="Evaluating Color", leave=False):
            x, y = x.to(DEVICE), y.to(DEVICE)
            pred = model(x)

            diff = pred - y
            total_abs_norm += diff.abs().sum().item()
            total_sq_norm += (diff ** 2).sum().item()
            total_pixels += y.numel()

            pred_orig = denormalize_rgb_tensor(pred)
            y_orig = denormalize_rgb_tensor(y)
            diff_orig = pred_orig - y_orig
            total_abs_orig += diff_orig.abs().sum().item()
            total_sq_orig += (diff_orig ** 2).sum().item()

    l1_norm = total_abs_norm / total_pixels
    mse_norm = total_sq_norm / total_pixels
    l1_orig = total_abs_orig / total_pixels
    mse_orig = total_sq_orig / total_pixels

    return {
        "l1_rgb_normalized": l1_norm,
        "mse_rgb_normalized": mse_norm,
        "psnr_rgb_normalized": psnr_from_mse(mse_norm, data_range=1.0),
        "l1_rgb_original_scale": l1_orig,
        "mse_rgb_original_scale": mse_orig
    }

def train_image_regression(model, train_loader, val_loader, task, epochs, lr, save_name, model_name, grad_clip=1.0):
    model = model.to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = nn.L1Loss()

    use_amp = (DEVICE == "cuda")
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    best_val = float("inf")
    save_path = os.path.join(SAVE_DIR, save_name)
    history = {"train_l1": [], "val_l1": [], "best_val_l1": None}

    for epoch in range(1, epochs + 1):
        model.train()
        train_sum = 0.0
        pbar = tqdm(train_loader, desc=f"{model_name} Epoch {epoch}/{epochs}", leave=False)

        for x, y, names in pbar:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=use_amp):
                pred = model(x)
                loss = criterion(pred, y)

            scaler.scale(loss).backward()

            if grad_clip is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

            scaler.step(optimizer)
            scaler.update()

            train_sum += loss.item() * x.size(0)
            pbar.set_postfix({"loss": f"{loss.item():.5f}"})

        train_l1 = train_sum / len(train_loader.dataset)

        model.eval()
        val_sum = 0.0
        with torch.no_grad():
            for x, y, names in val_loader:
                x = x.to(DEVICE, non_blocking=True)
                y = y.to(DEVICE, non_blocking=True)
                pred = model(x)
                loss = criterion(pred, y)
                val_sum += loss.item() * x.size(0)

        val_l1 = val_sum / len(val_loader.dataset)

        history["train_l1"].append(train_l1)
        history["val_l1"].append(val_l1)

        extra_text = f" | Val MAE≈{val_l1 * TIR_STD:.3f} K" if task == "sr" else ""
        print(f"Epoch {epoch:02d}/{epochs} | Train L1: {train_l1:.6f} | Val L1: {val_l1:.6f}{extra_text}")

        if val_l1 < best_val:
            best_val = val_l1
            history["best_val_l1"] = best_val
            save_checkpoint(
                model, save_path, model_name, task, history,
                extra={"best_val_l1": best_val}
            )
            print("  saved best checkpoint:", save_path)

    return history, save_path

In [ ]:
# ============================================================
# 10. Train CNN Super-Resolution model
# ============================================================

sr_model = SimpleSRNet(channels=64, num_blocks=6).to(DEVICE)

sr_history, sr_ckpt_path = train_image_regression(
    model=sr_model,
    train_loader=sr_train_loader,
    val_loader=sr_val_loader,
    task="sr",
    epochs=SR_EPOCHS,
    lr=1e-4,
    save_name="sr_cnn_residual_original_sensor_values.pth",
    model_name="SimpleSRNet"
)

print("Best SR checkpoint:", sr_ckpt_path)

sr_model = load_model_checkpoint(SimpleSRNet(channels=64, num_blocks=6), sr_ckpt_path)
sr_test_metrics = evaluate_sr(sr_model, sr_test_loader)
print("SR test metrics:")
for k, v in sr_test_metrics.items():
    print(f"  {k}: {v:.6f}")

In [ ]:
# ============================================================
# 11. Visualize SR prediction
# ============================================================

def visualize_sr_prediction(model, ds, idx=0):
    model.eval()
    x, y, name = ds[idx]

    with torch.no_grad():
        pred = model(x.unsqueeze(0).to(DEVICE)).cpu()[0]

    x_raw = denormalize_tir_tensor(x)[0].numpy()
    y_raw = denormalize_tir_tensor(y)[0].numpy()
    p_raw = denormalize_tir_tensor(pred)[0].numpy()
    abs_error = np.abs(p_raw - y_raw)

    plt.figure(figsize=(16, 4))

    plt.subplot(1, 4, 1)
    plt.imshow(x_raw, cmap="inferno")
    plt.title(f"Input 200m\n{x_raw.shape}")
    plt.colorbar()
    plt.axis("off")

    plt.subplot(1, 4, 2)
    plt.imshow(y_raw, cmap="inferno")
    plt.title(f"Target 100m\n{y_raw.shape}")
    plt.colorbar()
    plt.axis("off")

    plt.subplot(1, 4, 3)
    plt.imshow(p_raw, cmap="inferno")
    plt.title(f"Predicted 100m\n{p_raw.shape}")
    plt.colorbar()
    plt.axis("off")

    plt.subplot(1, 4, 4)
    plt.imshow(abs_error, cmap="magma")
    plt.title(f"Abs Error\nMean {abs_error.mean():.3f} K")
    plt.colorbar()
    plt.axis("off")

    plt.show()

    print("File:", name)
    print("Mean absolute error:", abs_error.mean(), "K")
    print("Pred raw range:", p_raw.min(), p_raw.max())

visualize_sr_prediction(sr_model, sr_val_ds, idx=0)

In [ ]:
# ============================================================
# 12. Train CNN U-Net Colorization model
# ============================================================

color_cnn_model = ColorUNet(base=32).to(DEVICE)

color_cnn_history, color_cnn_ckpt_path = train_image_regression(
    model=color_cnn_model,
    train_loader=color_train_loader,
    val_loader=color_val_loader,
    task="colorization",
    epochs=COLOR_CNN_EPOCHS,
    lr=1e-4,
    save_name="color_cnn_unet_original_sensor_values.pth",
    model_name="ColorUNet"
)

print("Best color CNN checkpoint:", color_cnn_ckpt_path)

color_cnn_model = load_model_checkpoint(ColorUNet(base=32), color_cnn_ckpt_path)
color_cnn_test_metrics = evaluate_color(color_cnn_model, color_test_loader)
print("Color CNN test metrics:")
for k, v in color_cnn_test_metrics.items():
    print(f"  {k}: {v:.6f}")

In [ ]:
# ============================================================
# 13. Visualize colorization prediction
# ============================================================

def visualize_color_prediction(model, ds, idx=0, title="Color Model"):
    model.eval()
    x, y, name = ds[idx]

    with torch.no_grad():
        pred = model(x.unsqueeze(0).to(DEVICE)).cpu()[0]

    x_raw = denormalize_tir_tensor(x)[0].numpy()
    y_raw = denormalize_rgb_tensor(y).numpy()
    p_raw = denormalize_rgb_tensor(pred).numpy()

    y_disp = rgb_chw_to_display(y_raw)
    p_disp = rgb_chw_to_display(p_raw)
    err = np.abs(p_raw - y_raw).mean(axis=0)

    plt.figure(figsize=(16, 4))

    plt.subplot(1, 4, 1)
    plt.imshow(x_raw, cmap="inferno")
    plt.title("Input TIR")
    plt.colorbar()
    plt.axis("off")

    plt.subplot(1, 4, 2)
    plt.imshow(y_disp)
    plt.title("Target RGB\n(display only)")
    plt.axis("off")

    plt.subplot(1, 4, 3)
    plt.imshow(p_disp)
    plt.title(f"{title}\n(display only)")
    plt.axis("off")

    plt.subplot(1, 4, 4)
    plt.imshow(err, cmap="magma")
    plt.title(f"Mean RGB Abs Error\n{err.mean():.6f}")
    plt.colorbar()
    plt.axis("off")

    plt.show()

    print("File:", name)
    print("Target original-scale range:", y_raw.min(), y_raw.max())
    print("Pred original-scale range:", p_raw.min(), p_raw.max())
    print("Mean original-scale RGB abs error:", err.mean())

visualize_color_prediction(color_cnn_model, color_val_ds, idx=0, title="CNN U-Net")

# 14. Model C — Pix2Pix GAN Colorization

GAN is optional/comparison. The CNN U-Net is the safest final colorization model.

Generator:

```text
TIR -> RGB
```

Discriminator:

```text
checks paired (TIR, RGB) real/fake patches
```

Generator loss:

```text
GAN loss + 100 × L1 loss
```

In [ ]:
# ============================================================
# 14. Pix2Pix GAN Colorization
# ============================================================

class PatchDiscriminator(nn.Module):
    def __init__(self, in_channels=4, base=64):
        super().__init__()

        def block(in_ch, out_ch, stride=2, use_bn=True):
            layers = [nn.Conv2d(in_ch, out_ch, 4, stride=stride, padding=1)]
            if use_bn:
                layers.append(nn.BatchNorm2d(out_ch))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers

        layers = []
        layers += block(in_channels, base, stride=2, use_bn=False)
        layers += block(base, base * 2, stride=2)
        layers += block(base * 2, base * 4, stride=2)
        layers += block(base * 4, base * 8, stride=1)
        layers += [nn.Conv2d(base * 8, 1, 4, stride=1, padding=1)]
        self.net = nn.Sequential(*layers)

    def forward(self, tir, rgb):
        return self.net(torch.cat([tir, rgb], dim=1))

def save_gan_checkpoint(G, D, path, history, best_val_l1):
    torch.save({
        "G_state_dict": G.state_dict(),
        "D_state_dict": D.state_dict(),
        "model_name": "Pix2Pix_ColorUNet_Generator_PatchDiscriminator",
        "task": "colorization_gan",
        "history": history,
        "best_val_l1": best_val_l1,
        "preprocess_stats": PREPROCESS_STATS,
        "created_time": time.strftime("%Y-%m-%d %H:%M:%S")
    }, path)

def train_pix2pix(G, D, train_loader, val_loader, epochs, lr=2e-4, lambda_l1=100.0,
                  save_name="color_pix2pix_original_sensor_values.pth"):
    G, D = G.to(DEVICE), D.to(DEVICE)
    opt_G = optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_D = optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))

    bce = nn.BCEWithLogitsLoss()
    l1 = nn.L1Loss()

    history = {"G_loss": [], "D_loss": [], "val_l1": []}
    best_val = float("inf")
    save_path = os.path.join(SAVE_DIR, save_name)

    for epoch in range(1, epochs + 1):
        G.train()
        D.train()
        g_sum, d_sum = 0.0, 0.0

        pbar = tqdm(train_loader, desc=f"Pix2Pix Epoch {epoch}/{epochs}", leave=False)

        for tir, real_rgb, names in pbar:
            tir, real_rgb = tir.to(DEVICE), real_rgb.to(DEVICE)

            # Train D
            with torch.no_grad():
                fake_rgb = G(tir)

            real_logits = D(tir, real_rgb)
            fake_logits = D(tir, fake_rgb.detach())

            d_loss_real = bce(real_logits, torch.ones_like(real_logits))
            d_loss_fake = bce(fake_logits, torch.zeros_like(fake_logits))
            d_loss = 0.5 * (d_loss_real + d_loss_fake)

            opt_D.zero_grad(set_to_none=True)
            d_loss.backward()
            opt_D.step()

            # Train G
            fake_rgb = G(tir)
            fake_logits = D(tir, fake_rgb)
            gan_loss = bce(fake_logits, torch.ones_like(fake_logits))
            recon_loss = l1(fake_rgb, real_rgb)
            g_loss = gan_loss + lambda_l1 * recon_loss

            opt_G.zero_grad(set_to_none=True)
            g_loss.backward()
            torch.nn.utils.clip_grad_norm_(G.parameters(), 1.0)
            opt_G.step()

            g_sum += g_loss.item() * tir.size(0)
            d_sum += d_loss.item() * tir.size(0)
            pbar.set_postfix({"G": f"{g_loss.item():.3f}", "D": f"{d_loss.item():.3f}", "L1": f"{recon_loss.item():.5f}"})

        train_g = g_sum / len(train_loader.dataset)
        train_d = d_sum / len(train_loader.dataset)

        G.eval()
        val_l1_sum = 0.0
        with torch.no_grad():
            for tir, real_rgb, names in val_loader:
                tir, real_rgb = tir.to(DEVICE), real_rgb.to(DEVICE)
                fake_rgb = G(tir)
                val_l1_sum += l1(fake_rgb, real_rgb).item() * tir.size(0)

        val_l1 = val_l1_sum / len(val_loader.dataset)

        history["G_loss"].append(train_g)
        history["D_loss"].append(train_d)
        history["val_l1"].append(val_l1)

        print(f"Epoch {epoch:02d}/{epochs} | G: {train_g:.5f} | D: {train_d:.5f} | Val L1: {val_l1:.6f}")

        if val_l1 < best_val:
            best_val = val_l1
            save_gan_checkpoint(G, D, save_path, history, best_val)
            print("  saved best GAN checkpoint:", save_path)

    return history, save_path

# Shape test
G_test = ColorUNet(base=32).to(DEVICE)
D_test = PatchDiscriminator(in_channels=4, base=64).to(DEVICE)
xb, yb, names = next(iter(color_train_loader))
xb, yb = xb.to(DEVICE), yb.to(DEVICE)
with torch.no_grad():
    fake = G_test(xb)
    logits = D_test(xb, fake)
print("GAN generator output:", fake.shape)
print("GAN discriminator logits:", logits.shape)

In [ ]:
# ============================================================
# 15. Train Pix2Pix GAN Colorization
# ============================================================

G = ColorUNet(base=32).to(DEVICE)
D = PatchDiscriminator(in_channels=4, base=64).to(DEVICE)

# Warm-start from CNN colorizer if available.
if os.path.exists(color_cnn_ckpt_path):
    ckpt = torch.load(color_cnn_ckpt_path, map_location=DEVICE)
    G.load_state_dict(ckpt["model_state_dict"])
    print("Warm-started GAN generator from CNN U-Net checkpoint.")

gan_history, gan_ckpt_path = train_pix2pix(
    G=G,
    D=D,
    train_loader=color_train_loader,
    val_loader=color_val_loader,
    epochs=GAN_EPOCHS,
    lr=2e-4,
    lambda_l1=100.0,
    save_name="color_pix2pix_original_sensor_values.pth"
)

print("Best GAN checkpoint:", gan_ckpt_path)

G_gan = ColorUNet(base=32).to(DEVICE)
ckpt = torch.load(gan_ckpt_path, map_location=DEVICE)
G_gan.load_state_dict(ckpt["G_state_dict"])
G_gan.eval()

gan_test_metrics = evaluate_color(G_gan, color_test_loader)
print("GAN generator test metrics:")
for k, v in gan_test_metrics.items():
    print(f"  {k}: {v:.6f}")

visualize_color_prediction(G_gan, color_val_ds, idx=1, title="Pix2Pix GAN")

# 16. Model D — Tiny Transformer Colorization

Objective:

```text
TIR 100m -> RGB 100m
```

Architecture:

```text
TIR -> patch embedding -> transformer encoder -> CNN decoder -> RGB
```

In [ ]:
# ============================================================
# 16. Tiny Transformer Colorization
# ============================================================

class TinyViTColorNet(nn.Module):
    def __init__(self, dim=128, depth=4, heads=4, patch=16):
        super().__init__()
        self.dim = dim
        self.patch = patch

        self.patch_embed = nn.Conv2d(1, dim, kernel_size=patch, stride=patch)
        self.num_tokens = (256 // patch) * (256 // patch)
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_tokens, dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=dim,
            nhead=heads,
            dim_feedforward=dim * 4,
            dropout=0.1,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)

        self.decoder = nn.Sequential(
            nn.Conv2d(dim, 128, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),

            nn.Conv2d(128, 96, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),

            nn.Conv2d(96, 64, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),

            nn.Conv2d(64, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),

            nn.Conv2d(32, 3, 3, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        feat = self.patch_embed(x)  # B,dim,16,16
        B, C, H, W = feat.shape
        tokens = feat.flatten(2).transpose(1, 2)
        tokens = tokens + self.pos_embed[:, :tokens.size(1), :]
        tokens = self.transformer(tokens)
        feat = tokens.transpose(1, 2).reshape(B, C, H, W)
        return self.decoder(feat)

vit_color_model = TinyViTColorNet(dim=128, depth=4, heads=4, patch=16).to(DEVICE)
xb, yb, names = next(iter(color_train_loader))
xb, yb = xb.to(DEVICE), yb.to(DEVICE)
with torch.no_grad():
    pred = vit_color_model(xb)

print("Input shape :", xb.shape)
print("Target shape:", yb.shape)
print("Pred shape  :", pred.shape)
print("Parameters  :", count_params(vit_color_model))
assert pred.shape == yb.shape

In [ ]:
# ============================================================
# 17. Train Tiny Transformer Colorization
# ============================================================

vit_color_model = TinyViTColorNet(dim=128, depth=4, heads=4, patch=16).to(DEVICE)

vit_history, vit_ckpt_path = train_image_regression(
    model=vit_color_model,
    train_loader=color_train_loader,
    val_loader=color_val_loader,
    task="colorization",
    epochs=VIT_EPOCHS,
    lr=1e-4,
    save_name="color_tiny_transformer_original_sensor_values.pth",
    model_name="TinyViTColorNet"
)

print("Best Transformer checkpoint:", vit_ckpt_path)

vit_color_model = load_model_checkpoint(TinyViTColorNet(dim=128, depth=4, heads=4, patch=16), vit_ckpt_path)
vit_test_metrics = evaluate_color(vit_color_model, color_test_loader)
print("Tiny Transformer test metrics:")
for k, v in vit_test_metrics.items():
    print(f"  {k}: {v:.6f}")

visualize_color_prediction(vit_color_model, color_val_ds, idx=2, title="Tiny Transformer")

In [ ]:
# ============================================================
# 18. Compare saved models
# ============================================================

comparison = {}

sr_path = os.path.join(SAVE_DIR, "sr_cnn_residual_original_sensor_values.pth")
if os.path.exists(sr_path):
    m = load_model_checkpoint(SimpleSRNet(channels=64, num_blocks=6), sr_path)
    comparison["SR CNN"] = evaluate_sr(m, sr_test_loader)

color_cnn_path = os.path.join(SAVE_DIR, "color_cnn_unet_original_sensor_values.pth")
if os.path.exists(color_cnn_path):
    m = load_model_checkpoint(ColorUNet(base=32), color_cnn_path)
    comparison["Color CNN U-Net"] = evaluate_color(m, color_test_loader)

gan_path = os.path.join(SAVE_DIR, "color_pix2pix_original_sensor_values.pth")
if os.path.exists(gan_path):
    m = ColorUNet(base=32).to(DEVICE)
    ckpt = torch.load(gan_path, map_location=DEVICE)
    m.load_state_dict(ckpt["G_state_dict"])
    m.eval()
    comparison["Color Pix2Pix GAN"] = evaluate_color(m, color_test_loader)

vit_path = os.path.join(SAVE_DIR, "color_tiny_transformer_original_sensor_values.pth")
if os.path.exists(vit_path):
    m = load_model_checkpoint(TinyViTColorNet(dim=128, depth=4, heads=4, patch=16), vit_path)
    comparison["Color Tiny Transformer"] = evaluate_color(m, color_test_loader)

print("\nMODEL COMPARISON")
for name, metrics in comparison.items():
    print("\n" + name)
    for k, v in metrics.items():
        print(f"  {k}: {v:.6f}")

metrics_path = os.path.join(SAVE_DIR, "model_comparison_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(comparison, f, indent=2)
print("\nSaved comparison metrics:", metrics_path)

# 19. Final Two-Stage Inference

Pipeline:

```text
Raw TIR 200m 256×256
        ↓
CNN SR
        ↓
Predicted TIR 100m 512×512
        ↓
Split into four 256×256 tiles
        ↓
Colorization model
        ↓
Merge tiles
        ↓
RGB-like 512×512 output
```

In [ ]:
# ============================================================
# 19. Final two-stage inference
# ============================================================

def predict_sr_from_raw_array(sr_model, tir_200m_hw):
    assert tir_200m_hw.shape == (256, 256), f"Expected 256x256 input, got {tir_200m_hw.shape}"
    sr_model.eval()

    x = torch.from_numpy(tir_200m_hw.astype(np.float32)).unsqueeze(0).unsqueeze(0)
    x = normalize_tir_tensor(x).to(DEVICE)

    with torch.no_grad():
        pred_norm = sr_model(x).cpu()[0]

    pred_raw = denormalize_tir_tensor(pred_norm)[0].numpy()
    return pred_raw.astype(np.float32)

def colorize_512_tir_by_tiles(color_model, tir_100m_512_hw):
    assert tir_100m_512_hw.shape == (512, 512), f"Expected 512x512, got {tir_100m_512_hw.shape}"
    color_model.eval()
    out_rgb_norm = np.zeros((3, 512, 512), dtype=np.float32)

    tiles = [
        (0, 256, 0, 256),
        (0, 256, 256, 512),
        (256, 512, 0, 256),
        (256, 512, 256, 512),
    ]

    with torch.no_grad():
        for r1, r2, c1, c2 in tiles:
            tile = tir_100m_512_hw[r1:r2, c1:c2].astype(np.float32)
            x = torch.from_numpy(tile).unsqueeze(0).unsqueeze(0)
            x = normalize_tir_tensor(x).to(DEVICE)
            pred_norm = color_model(x).cpu()[0]
            out_rgb_norm[:, r1:r2, c1:c2] = pred_norm.numpy()

    out_rgb_tensor = torch.from_numpy(out_rgb_norm)
    out_rgb_raw = denormalize_rgb_tensor(out_rgb_tensor).numpy()
    return out_rgb_raw.astype(np.float32)

def save_final_outputs(out_prefix, pred_tir_100m_512, pred_rgb_chw):
    os.makedirs(os.path.dirname(out_prefix), exist_ok=True)

    tir_path = out_prefix + "_pred_tir100m_512.npy"
    rgb_path = out_prefix + "_pred_rgb_chw_original_scale.npy"
    np.save(tir_path, pred_tir_100m_512.astype(np.float32))
    np.save(rgb_path, pred_rgb_chw.astype(np.float32))

    print("Saved:", tir_path)
    print("Saved:", rgb_path)

    tif_path = None
    try:
        import tifffile
        # Save BGR order if required by final project convention.
        bgr_chw = pred_rgb_chw[[2, 1, 0], :, :]
        tif_path = out_prefix + "_pred_bgr_chw.tif"
        tifffile.imwrite(tif_path, bgr_chw.astype(np.float32))
        print("Saved BGR TIFF:", tif_path)
    except Exception as e:
        print("TIFF save skipped:", e)

    preview_path = out_prefix + "_preview.png"
    try:
        rgb_disp = rgb_chw_to_display(pred_rgb_chw)
        plt.figure(figsize=(5, 5))
        plt.imshow(rgb_disp)
        plt.axis("off")
        plt.title("Preview only: stretched for display")
        plt.tight_layout()
        plt.savefig(preview_path, dpi=160)
        plt.close()
        print("Saved preview PNG:", preview_path)
    except Exception as e:
        print("Preview save skipped:", e)

    return {"tir_npy": tir_path, "rgb_npy": rgb_path, "tif": tif_path, "preview": preview_path}

def load_best_color_model(choice="cnn"):
    if choice == "cnn":
        path = os.path.join(SAVE_DIR, "color_cnn_unet_original_sensor_values.pth")
        return load_model_checkpoint(ColorUNet(base=32), path)

    if choice == "gan":
        path = os.path.join(SAVE_DIR, "color_pix2pix_original_sensor_values.pth")
        model = ColorUNet(base=32).to(DEVICE)
        ckpt = torch.load(path, map_location=DEVICE)
        model.load_state_dict(ckpt["G_state_dict"])
        model.eval()
        return model

    if choice == "transformer":
        path = os.path.join(SAVE_DIR, "color_tiny_transformer_original_sensor_values.pth")
        return load_model_checkpoint(TinyViTColorNet(dim=128, depth=4, heads=4, patch=16), path)

    raise ValueError("choice must be 'cnn', 'gan', or 'transformer'")

sr_model = load_model_checkpoint(
    SimpleSRNet(channels=64, num_blocks=6),
    os.path.join(SAVE_DIR, "sr_cnn_residual_original_sensor_values.pth")
)

COLOR_MODEL_CHOICE = "cnn"  # "cnn", "gan", or "transformer"
final_color_model = load_best_color_model(COLOR_MODEL_CHOICE)

sample_name = sr_test_ds.names[0]
tir_200m_path = os.path.join(DATASET_ROOT, "sr", "test", "tir_200m", sample_name)
tir_200m_raw = safe_load_npy(tir_200m_path)

pred_tir_100m_512 = predict_sr_from_raw_array(sr_model, tir_200m_raw)
pred_rgb_512 = colorize_512_tir_by_tiles(final_color_model, pred_tir_100m_512)

out_dir = os.path.join(SAVE_DIR, "final_two_stage_outputs")
out_prefix = os.path.join(out_dir, sample_name.replace(".npy", f"_{COLOR_MODEL_CHOICE}"))
saved_paths = save_final_outputs(out_prefix, pred_tir_100m_512, pred_rgb_512)

plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.imshow(tir_200m_raw, cmap="inferno")
plt.title("Raw input TIR 200m\n256×256")
plt.colorbar()
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(pred_tir_100m_512, cmap="inferno")
plt.title("Predicted TIR 100m\n512×512")
plt.colorbar()
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(rgb_chw_to_display(pred_rgb_512))
plt.title(f"Predicted RGB-like\n512×512 ({COLOR_MODEL_CHOICE})")
plt.axis("off")
plt.show()

print("Sample:", sample_name)
print("Predicted TIR range:", pred_tir_100m_512.min(), pred_tir_100m_512.max())
print("Predicted RGB original-scale range:", pred_rgb_512.min(), pred_rgb_512.max())
print("Saved paths:", saved_paths)

# 20. Batch Inference on Common Hackathon Dataset

Use this if organizers provide a common folder of raw `.npy` TIR 200m files.

Expected input:

```text
common_dataset/
  sample_001.npy
  sample_002.npy
```

Each `.npy` file should be:

```text
shape = (256, 256)
raw original TIR sensor values
```

In [ ]:
# ============================================================
# 20. Batch inference for common evaluation folder
# ============================================================

def run_batch_inference_on_folder(input_folder, output_folder, color_choice="cnn", max_files=None):
    assert os.path.exists(input_folder), f"Input folder not found: {input_folder}"
    os.makedirs(output_folder, exist_ok=True)

    sr_path = os.path.join(SAVE_DIR, "sr_cnn_residual_original_sensor_values.pth")
    sr_model_local = load_model_checkpoint(SimpleSRNet(channels=64, num_blocks=6), sr_path)
    color_model_local = load_best_color_model(color_choice)

    files = sorted(glob.glob(os.path.join(input_folder, "*.npy")))
    if max_files is not None:
        files = files[:max_files]

    assert len(files) > 0, f"No .npy files found in {input_folder}"

    records = []

    for p in tqdm(files, desc="Batch inference"):
        name = os.path.basename(p)
        raw = safe_load_npy(p)

        if raw.shape != (256, 256):
            print(f"Skipping {name}: expected (256,256), got {raw.shape}")
            continue

        pred_tir = predict_sr_from_raw_array(sr_model_local, raw)
        pred_rgb = colorize_512_tir_by_tiles(color_model_local, pred_tir)

        prefix = os.path.join(output_folder, name.replace(".npy", f"_{color_choice}"))
        saved = save_final_outputs(prefix, pred_tir, pred_rgb)

        records.append({"input": p, "output_prefix": prefix, **saved})

    manifest_path = os.path.join(output_folder, "inference_manifest.json")
    with open(manifest_path, "w") as f:
        json.dump(records, f, indent=2)

    print("Batch inference complete.")
    print("Manifest:", manifest_path)
    return records

# Example during finale:
# COMMON_INPUT_FOLDER = "/content/common_dataset/tir_200m"
# COMMON_OUTPUT_FOLDER = "/content/drive/MyDrive/landsat_india_200/common_outputs"
# run_batch_inference_on_folder(COMMON_INPUT_FOLDER, COMMON_OUTPUT_FOLDER, color_choice="cnn")

# Final Notes for PPT / Submission

Use this statement clearly:

> We do not train on visually enhanced or pseudo-colored images. The model uses original Landsat-9 sensor arrays, and visualization stretching is used only for human-readable plots.

Recommended final model for safety:

```text
CNN SR + CNN U-Net Colorization
```

GAN and Transformer are included as comparative/extension models.

Generated deliverables:

```text
preprocess_stats.json
sr_cnn_residual_original_sensor_values.pth
color_cnn_unet_original_sensor_values.pth
color_pix2pix_original_sensor_values.pth
color_tiny_transformer_original_sensor_values.pth
model_comparison_metrics.json
final_two_stage_outputs/
```

Checklist before submission:

- Dataset sanity check passes
- Preprocessing stats saved
- SR model trains and visualizes correctly
- Color CNN trains and visualizes correctly
- Final two-stage inference runs
- PPT mentions original sensor value handling
- No PNG/colormap/visualization image is used as model input